In [2]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

#### Creating the dataset for modeling

In [4]:
# Load data
trips    = pd.read_csv('../../data/data/processed/trips.csv')
weather  = pd.read_csv('../../data/data/processed/weather.csv')
stations = pd.read_csv('../../data/data/processed/stations.csv')

In [5]:
# Aggregate trip counts per station-hour
demand = (
    trips.groupby(['start_station', 'st_year', 'st_month', 'st_day', 'st_hour'])
         .size()
         .reset_index(name='trip_count')
)

In [6]:

demand['timestamp'] = pd.to_datetime(
    demand[['st_year', 'st_month', 'st_day', 'st_hour']]
    .rename(columns={'st_year': 'year', 'st_month': 'month', 'st_day': 'day', 'st_hour': 'hour'})
)

expand to full station x hour timegrid

In [7]:
all_hours      = pd.date_range(demand['timestamp'].min(), demand['timestamp'].max(), freq='h')
stations_list  = trips['start_station'].unique()

In [8]:
fdemand = (
    pd.MultiIndex.from_product([stations_list, all_hours], names=['start_station', 'timestamp'])
    .to_frame(index=False)
    .merge(demand[['start_station', 'timestamp', 'trip_count']],
           on=['start_station', 'timestamp'], how='left')
)
fdemand['trip_count'] = fdemand['trip_count'].fillna(0).astype(int)

In [9]:
# Datetime features
fdemand['st_year']     = fdemand['timestamp'].dt.year
fdemand['st_month']    = fdemand['timestamp'].dt.month
fdemand['st_day']      = fdemand['timestamp'].dt.day
fdemand['st_hour']     = fdemand['timestamp'].dt.hour
fdemand['day_of_week'] = fdemand['timestamp'].dt.dayofweek
fdemand['is_weekend']  = (fdemand['day_of_week'] >= 5).astype(int)

In [10]:
# Cyclical encodings
fdemand['hour_sin']  = np.sin(2 * np.pi * fdemand['st_hour']  / 24)
fdemand['hour_cos']  = np.cos(2 * np.pi * fdemand['st_hour']  / 24)
fdemand['month_sin'] = np.sin(2 * np.pi * fdemand['st_month'] / 12)
fdemand['month_cos'] = np.cos(2 * np.pi * fdemand['st_month'] / 12)

In [11]:
# Lag features (sort first)
fdemand = fdemand.sort_values(['start_station', 'timestamp'])
for lag, col in [(1, 'lag_1hr'), (24, 'lag_24hr'), (168, 'lag_168hr')]:
    fdemand[col] = fdemand.groupby('start_station')['trip_count'].shift(lag)

In [12]:
# Merge weather and stations
weather['date'] = pd.to_datetime(weather['date']).dt.date
fdemand['date'] = fdemand['timestamp'].dt.date
fdemand = (
    fdemand.merge(weather, on='date', how='left')
           .drop(columns='date')
           .merge(stations, left_on='start_station', right_on='id', how='left')
)

In [16]:
fdemand.tail()

,start_station,timestamp,trip_count,st_year,st_month,st_day,st_hour,day_of_week,is_weekend,hour_sin,...,snow,snow_ground,max_temp,min_temp,id,name,latitude,longitude,total_docks,street_address
18894955,3461,2026-03-31 19:00:00,0,2026,3,31,19,1,0,-0.965926,...,0.0,0.0,81,62,3461.0,"69th & Guyer, James Finnegan Playground",39.91532,-75.23281,13.0,2800 S 69th St
18894956,3461,2026-03-31 20:00:00,0,2026,3,31,20,1,0,-0.866025,...,0.0,0.0,81,62,3461.0,"69th & Guyer, James Finnegan Playground",39.91532,-75.23281,13.0,2800 S 69th St
18894957,3461,2026-03-31 21:00:00,0,2026,3,31,21,1,0,-0.707107,...,0.0,0.0,81,62,3461.0,"69th & Guyer, James Finnegan Playground",39.91532,-75.23281,13.0,2800 S 69th St
18894958,3461,2026-03-31 22:00:00,0,2026,3,31,22,1,0,-0.500000,...,0.0,0.0,81,62,3461.0,"69th & Guyer, James Finnegan Playground",39.91532,-75.23281,13.0,2800 S 69th St
18894959,3461,2026-03-31 23:00:00,0,2026,3,31,23,1,0,-0.258819,...,0.0,0.0,81,62,3461.0,"69th & Guyer, James Finnegan Playground",39.91532,-75.23281,13.0,2800 S 69th St


In [14]:
fdemand.dtypes

start_station              int64
timestamp         datetime64[us]
trip_count                 int64
st_year                    int32
st_month                   int32
st_day                     int32
st_hour                    int32
day_of_week                int32
is_weekend                 int64
hour_sin                 float64
hour_cos                 float64
month_sin                float64
month_cos                float64
lag_1hr                  float64
lag_24hr                 float64
lag_168hr                float64
avg_wind                 float64
precip                   float64
snow                     float64
snow_ground              float64
max_temp                   int64
min_temp                   int64
id                       float64
name                         str
latitude                 float64
longitude                float64
total_docks              float64
street_address               str
dtype: object

In [15]:
# don't run this yet, but create a feature list and potentially drop later
#df = fdemand.drop(['timestamp', 'name', 'street_address', 'id', 'st_hour', 'st_month'], axis=1)

### Time for models??

In [20]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

 quick note: these models are trained on a time-based split, not a train_test_split()

In [21]:
model_df = fdemand.copy()

# Remove rows with missing lag values
model_df = model_df.dropna()

In [22]:
features = [
    "st_year",
    "day_of_week",
    "is_weekend",

    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",

    "avg_wind",
    "precip",
    "snow",
    "snow_ground",
    "max_temp",
    "min_temp",

    "latitude",
    "longitude",
    "total_docks",

    "lag_1hr",
    "lag_24hr",
    "lag_168hr"
]

In [23]:
target = 'trip_count'

splitting the data into train/test

In [24]:
split_date = "2025-01-01"

train = model_df[
    model_df["timestamp"] < split_date
]

test = model_df[
    model_df["timestamp"] >= split_date
]

In [25]:
X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

train linear regression

In [26]:
lr = LinearRegression()

lr.fit(X_train, y_train)

preds = lr.predict(X_test)

# Trip counts can't be negative
preds = np.maximum(preds, 0)

In [27]:
mae = mean_absolute_error(y_test, preds)

rmse = np.sqrt(
    mean_squared_error(y_test, preds)
)

r2 = r2_score(y_test, preds)

print("\nLinear Regression Results")
print("-" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


Linear Regression Results
----------------------------------------
MAE  : 0.4505
RMSE : 0.8182
R²   : 0.3827


In [28]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": lr.coef_
})

coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values(
    "Abs_Coefficient",
    ascending=False
)

print("\nTop Features")
print("-" * 40)
print(coef_df[["Feature", "Coefficient"]].head(15))


Top Features
----------------------------------------
        Feature  Coefficient
13     latitude    -0.513496
14    longitude     0.281550
18    lag_168hr     0.261052
16      lag_1hr     0.249509
17     lag_24hr     0.236442
8        precip    -0.065413
4      hour_cos    -0.062707
3      hour_sin    -0.034037
0       st_year     0.012204
2    is_weekend    -0.009559
5     month_sin    -0.009326
15  total_docks     0.007102
10  snow_ground    -0.004640
9          snow    -0.003062
1   day_of_week    -0.002659


In [29]:
print(model_df.shape)
print(model_df["trip_count"].describe())
print((model_df["trip_count"] == 0).mean())

(16161600, 28)
count    1.616160e+07
mean     3.385152e-01
std      9.556697e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.200000e+01
Name: trip_count, dtype: float64
0.8182574745074745


try a poisson model (chatgpt told me to)

In [30]:
from sklearn.linear_model import PoissonRegressor

In [31]:
poisson = PoissonRegressor(
    alpha=1.0,
    max_iter=1000
)

poisson.fit(X_train, y_train)

preds = poisson.predict(X_test)

In [33]:
mae = mean_absolute_error(y_test, preds)

rmse = np.sqrt(
    mean_squared_error(y_test, preds)
)

r2 = r2_score(y_test, preds)

print("\nPoisson Regression Results")
print("-" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


Poisson Regression Results
----------------------------------------
MAE  : 0.5207
RMSE : 1.0114
R²   : 0.0568


train random forest

In [34]:
from sklearn.ensemble import RandomForestRegressor

In [35]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

preds = rf.predict(X_test)

In [36]:
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

In [37]:
print("\nRandom Forest Results")
print("-" * 40)
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


Random Forest Results
----------------------------------------
MAE  : 0.4217
RMSE : 0.7857
R²   : 0.4308


lin reg using stats regression package

##### cut test and train to smaller date ranges

In [41]:
# Define new date ranges for training and testing data
train_start_date = '2023-01-01'
train_end_date = '2024-12-31'
test_start_date = '2025-01-01'
test_end_date = '2026-03-31'

In [43]:
# Split data into train and test sets based on the new date ranges

ntrain = model_df[
    (model_df['timestamp'] >= pd.to_datetime(train_start_date)) &
    (model_df['timestamp'] <= pd.to_datetime(train_end_date))
]

ntest = model_df[
    (model_df['timestamp'] >= pd.to_datetime(test_start_date)) &
    (model_df['timestamp'] <= pd.to_datetime(test_end_date))
]

print(f"Train data shape: {ntrain.shape}")
print(f"Test data shape: {ntest.shape}")

Train data shape: (5186216, 28)
Test data shape: (3225512, 28)


In [44]:
Xn_train = ntrain[features]
yn_train = ntrain[target]

Xn_test = ntest[features]
yn_test = ntest[target]

In [46]:
# Initialize and train the Linear Regression model
lr_model_new = LinearRegression()
lr_model_new.fit(Xn_train, yn_train)

print("New Linear Regression model trained successfully!")

# Make predictions on the test set
y_pred_lr_new = lr_model_new.predict(Xn_test)

# Evaluate the model
mse_lr_new = mean_squared_error(yn_test, y_pred_lr_new)
r2_lr_new = r2_score(yn_test, y_pred_lr_new)

print(f"New Linear Regression Mean Squared Error (MSE): {mse_lr_new:.4f}")
print(f"New Linear Regression R-squared (R2) Score: {r2_lr_new:.4f}")

New Linear Regression model trained successfully!
New Linear Regression Mean Squared Error (MSE): 0.6675
New Linear Regression R-squared (R2) Score: 0.3832


In [47]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Combine Xn_train and yn_train for statsmodels (it prefers a single DataFrame)
train_data_new = Xn_train.copy()
train_data_new[target] = yn_train

# Define the Poisson regression model
poisson_model_new = smf.glm(formula=f"{target} ~ {' + '.join(features)}",
                            data=train_data_new,
                            family=sm.families.Poisson()).fit()

print("New Poisson Regression model trained successfully!")

# Make predictions on the test set
y_pred_poisson_new = poisson_model_new.predict(Xn_test)

# Evaluate the model
mse_poisson_new = mean_squared_error(yn_test, y_pred_poisson_new)
r2_poisson_new = r2_score(yn_test, y_pred_poisson_new)

print(f"New Poisson Regression Mean Squared Error (MSE): {mse_poisson_new:.2f}")
print(f"New Poisson Regression R-squared (R2) Score: {r2_poisson_new:.2f}")

# Display a summary of the model (optional, but useful for diagnostics)
print(poisson_model_new.summary())

New Poisson Regression model trained successfully!
New Poisson Regression Mean Squared Error (MSE): 6.74
New Poisson Regression R-squared (R2) Score: -5.23
                 Generalized Linear Model Regression Results                  
Dep. Variable:             trip_count   No. Observations:              5186216
Model:                            GLM   Df Residuals:                  5186196
Model Family:                 Poisson   Df Model:                           19
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -3.6899e+06
Date:                Sun, 21 Jun 2026   Deviance:                   4.7328e+06
Time:                        12:33:15   Pearson chi2:                 6.69e+06
No. Iterations:                     8   Pseudo R-squ. (CS):             0.3768
Covariance Type:            nonrobust                                         
                  coef    std err          z      P>|z

In [48]:
from xgboost import XGBRegressor

# Initialize and train the XGBoost Regressor model
xgb_model_new = XGBRegressor(random_state=42) # Added random_state for reproducibility
xgb_model_new.fit(Xn_train, yn_train)

print("New XGBoost Regressor model trained successfully!")

# Make predictions on the test set
y_pred_xgb_new = xgb_model_new.predict(Xn_test)

# Evaluate the model
mse_xgb_new = mean_squared_error(yn_test, y_pred_xgb_new)
r2_xgb_new = r2_score(yn_test, y_pred_xgb_new)

print(f"New XGBoost Regressor Mean Squared Error (MSE): {mse_xgb_new:.2f}")
print(f"New XGBoost Regressor R-squared (R2) Score: {r2_xgb_new:.2f}")

New XGBoost Regressor model trained successfully!
New XGBoost Regressor Mean Squared Error (MSE): 0.60
New XGBoost Regressor R-squared (R2) Score: 0.45


variable importance

In [ ]:
import shap

In [51]:
# Create a SHAP explainer for the Poisson GLM model
# For statsmodels GLM, we can use shap.Explainer with the model's predict method
explainer = shap.Explainer(poisson_model_new.predict, Xn_train)

# Calculate SHAP values for the test set
# This might take a moment depending on the size of X_test
shap_values = explainer(Xn_test)

print("SHAP values calculated successfully!")

Background dataset has 5186216 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=5186216 when initializing the masker.
PermutationExplainer explainer:   0%|          | 10871/3225512 [55:26<273:56:34,  3.26it/s]


KeyboardInterrupt: 

In [ ]:
shap.summary_plot(shap_values, Xn_test, plot_type="bar", title="SHAP Feature Importance for Poisson Model")

# Plot the SHAP summary plot (beeswarm) to see distribution and impact direction
shap.summary_plot(shap_values, Xn_test, title="SHAP Feature Impact for Poisson Model")

In [3]:
fff = pd.read_csv('../../data/data/processed/trips.csv')

In [4]:
fff.shape

(6280562, 28)

Random sample of trip data for use in colab

In [5]:
sample_6000 = fff.sample(n=6000, random_state=42)

In [6]:
sample_6000.to_csv('sample_trip.csv', index=False)

In [17]:
sample_6000['start_station'].nunique()

303

In [15]:
sample_6000['end_station'].nunique()

301